# Sprint 1 — Data Pipeline (Kibera MVP)

Validates all four data sources (Sentinel-2, Sentinel-1, SRTM/TWI, OSM vectors), producing:
- a multi-band GeoTIFF raster stack exported to Drive
- a serialised PyTorch Geometric graph for the building/infrastructure network

per proposal Sec 3.4.3 (Sprint 1, weeks 1-2). Run cells top to bottom.

## 1. Environment setup

In [ ]:
# Clone the repo (replace with your actual GitHub URL once pushed)
REPO_URL = "https://github.com/<your-username>/nairobi-risk-mapping.git"
!git clone $REPO_URL
%cd nairobi-risk-mapping

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Mount Drive -- checkpoints and exported rasters need to survive session resets
from google.colab import drive
drive.mount('/content/drive')

## 2. Earth Engine authentication

Needs a GCP project with the Earth Engine API enabled. Create one at
https://console.cloud.google.com if you haven't already, then enable Earth
Engine at https://developers.google.com/earth-engine/guides/access.

In [ ]:
GEE_PROJECT_ID = "your-gcp-project-id"  # <-- set this

from src.utils import gee_utils
gee_utils.initialize_gee(GEE_PROJECT_ID)
print("Earth Engine initialised.")

## 3. Run the Sprint 1 pipeline for Kibera

In [ ]:
from src.data_pipeline import DataPipeline

pipeline = DataPipeline(settlement_key="kibera", gee_project_id=GEE_PROJECT_ID)

# Sanity-check the resolved boundary before spending GEE quota on it
boundary = pipeline.get_boundary()
boundary.plot()
boundary

In [ ]:
# Graph branch -- runs synchronously, few minutes for a single settlement
graph_data, integrity = pipeline.acquire_graph_data()
print(integrity)

In [ ]:
# Raster branch -- kicks off an async export to Drive, doesn't block
task = pipeline.acquire_raster_stack()
task.status()

In [ ]:
# Poll until the export finishes (check every 30s). This can take
# several minutes to an hour depending on area size and GEE queue load.
import time
while task.active():
    print(task.status()['state'])
    time.sleep(30)
print("Final status:", task.status())

## 4. Tile the exported raster stack

Once the export finishes, find the GeoTIFF in your Drive under
`nairobi_risk_mapping/kibera_raster_stack.tif` and point `tile_raster_stack`
at it.

In [ ]:
stack_path = "/content/drive/MyDrive/nairobi_risk_mapping/kibera_raster_stack.tif"
manifest_path = pipeline.tile_raster_stack(stack_path, split="train")
print("Tile manifest:", manifest_path)

## 5. Running Mathare and Mukuru too

`DataPipeline` takes `settlement_key` as a parameter and `config.SETTLEMENTS`
already defines all three settlements, so nothing above is Kibera-specific --
you can rerun steps 3-4 with `settlement_key="mathare"` or `"mukuru"` as-is.

For running all three back-to-back, use the batch helper instead:

In [ ]:
from src.data_pipeline import run_sprint1_all_settlements

# Kicks off graph acquisition (synchronous) + raster export (async, one GEE
# task per settlement) for Kibera, Mathare, and Mukuru in one call. A failure
# on one settlement (e.g. an Overpass timeout) doesn't block the other two --
# check summary[key]['status'] afterwards.
summary = run_sprint1_all_settlements(gee_project_id=GEE_PROJECT_ID)

for key, result in summary.items():
    print(key, "->", result.get("status"))
    if result.get("status") == "ok":
        print("   graph integrity:", result["graph_integrity"])

In [ ]:
# Each settlement's raster export is an independent async task -- poll all
# three and tile each once its export finishes.
for key, result in summary.items():
    if result.get("status") != "ok":
        continue
    task = result["raster_export_task"]
    print(key, task.status()["state"])

In [ ]:
# Once each export shows COMPLETED in Drive under nairobi_risk_mapping/,
# tile it the same way as the Kibera walkthrough above:
from src.data_pipeline import DataPipeline

for key in ["mathare", "mukuru"]:
    stack_path = f"/content/drive/MyDrive/nairobi_risk_mapping/{key}_raster_stack.tif"
    pipeline = DataPipeline(settlement_key=key, gee_project_id=GEE_PROJECT_ID)
    manifest_path = pipeline.tile_raster_stack(stack_path, split="train")
    print(key, "->", manifest_path)

## Sprint 1 exit criteria (proposal Sec 3.4.3)

- [ ] All four data sources acquired for Kibera, Mathare, and Mukuru without errors
- [ ] Raster stack exported and successfully tiled for each settlement (check discard rate isn't excessive)
- [ ] Graph integrity check shows a largest-connected-component fraction well above 0.5 for each settlement
- [ ] Manifest and graph `.pt` files present in `data/` for all three settlements

Next: Sprint 2 — Attention U-Net transfer-learning baseline on this tiled data.